# Memory Management with `CosyScope`

This tutorial explains how to correctly manage memory when using the **COSY backend** in Sandalwood.

## The Challenge: Stack Allocation

The COSY backend uses a high-performance **Stack Allocator** for memory. This is different from Python's Heap memory.

- **Python**: Objects are freed whenever the Garbage Collector runs (non-deterministic).
- **COSY**: Objects are allocated on a stack (LIFO). They must be freed in the reverse order of creation.

If Python's GC tries to free a variable that is *not at the top of the stack*, COSY cannot free it safely. This leads to **memory leaks** (holes in the stack) that persist until the program ends.

## The Solution: `CosyScope`

`CosyScope` is a context manager that allows you to define a block of code where temporary variables are created. When the block exits, `CosyScope` automatically **rewinds the stack**, instantly freeing all variables created inside it.


In [1]:
import os

import psutil

from sandalwood.backends.cosy.cosy_backend import CosyScope
from sandalwood.taylor_function import MultivariateTaylorFunction as MTF

# Ensure we are using COSY
MTF.initialize_mtf(max_order=2, max_dimension=2, implementation="cosy")
print(f"Backend: {MTF._IMPLEMENTATION}")

Initializing MTF globals with: _MAX_ORDER=2, _MAX_DIMENSION=2 using cosy backend
Initializing COSY backend...
MTF globals initialized: _MAX_ORDER=2, _MAX_DIMENSION=2, _INITIALIZED=True
Max coefficient count (order=2, nvars=2): 6
Backend: cosy


### Example 1: The Problem (Simulated)

If you run a long loop creating temporary variables *without* a scope, the COSY internal stack will grow indefinitely.

In [2]:
def print_memory():
    process = psutil.Process(os.getpid())
    print(f"RSS Memory: {process.memory_info().rss / 1024 / 1024:.2f} MB")


print("Before loop:")
print_memory()

# Imagine a loop that runs 100,000 times.
# Without CosyScope, each iteration leaves residue on the COSY stack because
# Python's GC doesn't perfectly align with COSY's stack discipline.
print("Running loop... (simulated small one for safety)")
for i in range(100):
    # Each call creates a new temporary on the stack
    x = MTF.var(1)
    y = x + 1.0
    # y goes out of scope here, but COSY might not reclaim it immediately/efficiently

print("After loop (Without Scope):")
print_memory()

Before loop:
RSS Memory: 650.38 MB
Running loop... (simulated small one for safety)
After loop (Without Scope):
RSS Memory: 650.44 MB


### Example 2: The Solution with `CosyScope`

By wrapping the loop body (or the entire loop if appropriate) in `CosyScope`, we guarantee that memory is reset.

In [3]:
print("Before Scoped loop:")
print_memory()

for i in range(100):
    with CosyScope():
        # Variables allocated here are temporary
        x = MTF.var(1)
        y = x + 1.0

        # Perform calculations...
        # If you need to save a result, extract it (e.g. to numpy or float)
        val = y.eval([0, 0])

    # Exiting scope: COSY stack is rewound. Memory usage stays constant.

print("After Scoped loop:")
print_memory()

Before Scoped loop:
RSS Memory: 650.46 MB


After Scoped loop:
RSS Memory: 666.64 MB


### Important Rules

1. **Do not use variables outside their scope.**
   Once the scope exits, the variables are invalid. Accessing them will likely crash or return garbage.

2. **Use Scopes for Looping.**
   Anytime you write a `for` or `while` loop that does math on MTF objects, put `with CosyScope():` inside instructions.

3. **Extract Results.**
   If you need to keep a result from a calculation inside a scope, calculate the final number or array and save that Python object, which is independent of COSY memory.


In [4]:
# Example of INVALID usage
saved_var = None
with CosyScope():
    x = MTF.var(1)
    saved_var = x + 2.0

# DANGER: saved_var now points to deallocated memory!
try:
    print(saved_var.eval([0, 0]))
except Exception as e:
    print(f"Caught expected error/crash: {e}")

[2.]
